# Preprocessing


## Setup

In [1]:
import pandas as pd
import numpy as np
import json
import os
import re
import ast

INTERIM_DIR = "../data/interim"
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

df_business = pd.read_json("../data/raw/yelp_json/yelp_academic_dataset_business.json", lines=True)
df_checkin  = pd.read_json("../data/raw/yelp_json/yelp_academic_dataset_checkin.json", lines=True)
df_tip      = pd.read_json("../data/raw/yelp_json/yelp_academic_dataset_tip.json", lines=True)
df_review   = pd.read_parquet(f"{INTERIM_DIR}/review.parquet")
df_user     = pd.read_parquet(f"{INTERIM_DIR}/user.parquet")

print(df_business.shape, df_checkin.shape, df_tip.shape, df_review.shape, df_user.shape)

(150346, 14) (131930, 2) (908915, 5) (6990280, 9) (1987897, 22)


## 1. Business Table: Deduplicate & Flatten Nested Columns


In [3]:
# Drop exact duplicate businesses (dedupe on business_id since some columns contain dicts)
df_business = df_business.drop_duplicates(subset=["business_id"])

# Flatten 'hours' dict into separate day columns
hours_df = df_business["hours"].apply(lambda x: x if isinstance(x, dict) else {})
hours_expanded = pd.json_normalize(hours_df)
hours_expanded.columns = [f"hours_{col}" for col in hours_expanded.columns]

# Flatten 'attributes' dict
attrs_df = df_business["attributes"].apply(lambda x: x if isinstance(x, dict) else {})
attrs_expanded = pd.json_normalize(attrs_df)
attrs_expanded.columns = [f"attr_{col}" for col in attrs_expanded.columns]

df_business_flat = pd.concat(
    [df_business.drop(columns=["hours", "attributes"]).reset_index(drop=True),
     hours_expanded.reset_index(drop=True),
     attrs_expanded.reset_index(drop=True)],
    axis=1
)

print(df_business_flat.shape)
df_business_flat.head()

(150346, 58)


,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,...,attr_AcceptsInsurance,attr_BestNights,attr_BYOB,attr_Corkage,attr_BYOBCorkage,attr_HairSpecializesIn,attr_Open24Hours,attr_RestaurantsCounterService,attr_AgesAllowed,attr_DietaryRestrictions
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Some attribute columns are stringified dicts themselves (e.g. "BusinessParking": "{'garage': False,...}")
# Check which columns need a second-level parse
def try_parse_dict(val):
    if isinstance(val, str) and val.startswith("{"):
        try:
            return ast.literal_eval(val)
        except (ValueError, SyntaxError):
            return val
    return val

nested_attr_cols = [col for col in attrs_expanded.columns
                     if attrs_expanded[col].astype(str).str.startswith("{").any()]
print("Columns needing second-level parsing:", nested_attr_cols)

Columns needing second-level parsing: ['attr_BusinessParking', 'attr_Ambience', 'attr_GoodForMeal', 'attr_Music', 'attr_BestNights', 'attr_HairSpecializesIn', 'attr_DietaryRestrictions']


In [6]:
# Parse and expand each nested attribute column
for col in nested_attr_cols:
    # Step 1: convert stringified dict -> real dict
    parsed = attrs_expanded[col].apply(try_parse_dict)
    
    # Step 2: expand dict into separate columns
    expanded = pd.json_normalize(parsed.apply(lambda x: x if isinstance(x, dict) else {}))
    expanded.columns = [f"{col}_{subcol}" for subcol in expanded.columns]
    
    # Step 3: attach back, drop original stringified column
    attrs_expanded = pd.concat([attrs_expanded.drop(columns=[col]), expanded], axis=1)

print("New shape after 2nd-level parsing:", attrs_expanded.shape)
attrs_expanded.head()

New shape after 2nd-level parsing: (150346, 81)


,attr_ByAppointmentOnly,attr_BusinessAcceptsCreditCards,attr_BikeParking,attr_RestaurantsPriceRange2,attr_CoatCheck,attr_RestaurantsTakeOut,attr_RestaurantsDelivery,attr_Caters,attr_WiFi,attr_WheelchairAccessible,...,attr_HairSpecializesIn_kids,attr_HairSpecializesIn_perms,attr_HairSpecializesIn_asian,attr_DietaryRestrictions_dairy-free,attr_DietaryRestrictions_gluten-free,attr_DietaryRestrictions_vegan,attr_DietaryRestrictions_kosher,attr_DietaryRestrictions_halal,attr_DietaryRestrictions_soy-free,attr_DietaryRestrictions_vegetarian
0,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,False,True,True,2,False,False,False,False,u'no',True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,False,False,True,1,NaN,True,False,True,u'free',NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,True,True,NaN,NaN,True,NaN,False,NaN,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
# These new sub-attribute columns will have NaNs where the parent attribute was missing
new_attr_cols = [c for c in attrs_expanded.columns if any(c.startswith
                                                          (f"{orig}_") for orig in nested_attr_cols)]
attrs_expanded[new_attr_cols] = attrs_expanded[new_attr_cols].fillna("Unknown")

## 2. Categories: Split into List

In [13]:
df_business_flat["categories_list"] = df_business_flat["categories"].apply(
    lambda x: x.split(", ") if isinstance(x, str) else []
)

## 3. Handle Missing Values

In [14]:
# Business: drop rows with no categories (not useful for most analyses)
df_business_flat = df_business_flat[df_business_flat["categories"].notna()]

# Fill missing attribute/hours columns
attr_cols = [c for c in df_business_flat.columns if c.startswith("attr_")]
df_business_flat[attr_cols] = df_business_flat[attr_cols].fillna("Unknown")

hours_cols = [c for c in df_business_flat.columns if c.startswith("hours_")]
df_business_flat[hours_cols] = df_business_flat[hours_cols].fillna("Closed")

## 4. Dtype Downcasting
Important for memory given the size of `review` and `user` tables.

In [15]:
def downcast_df(df):
    for col in df.select_dtypes(include=["int64"]).columns:
        df[col] = pd.to_numeric(df[col], downcast="integer")
    for col in df.select_dtypes(include=["float64"]).columns:
        df[col] = pd.to_numeric(df[col], downcast="float")
    return df

df_review = downcast_df(df_review)
df_user   = downcast_df(df_user)
df_business_flat = downcast_df(df_business_flat)

print("Review memory:", df_review.memory_usage(deep=True).sum() / 1e6, "MB")
print("User memory:", df_user.memory_usage(deep=True).sum() / 1e6, "MB")

Review memory: 6141.364502 MB
User memory: 3155.264608 MB


## 5. Date Conversions

In [16]:
df_review["date"] = pd.to_datetime(df_review["date"])
df_tip["date"] = pd.to_datetime(df_tip["date"])
df_user["yelping_since"] = pd.to_datetime(df_user["yelping_since"])

## 6. Text Cleaning
Light cleaning on `review` and `tip` text — keeps basic punctuation, strips URLs/noise.

In [17]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)   # remove URLs
    text = re.sub(r"[^a-z0-9\s.,!?']", " ", text) # keep basic punctuation
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_review["clean_text"] = df_review["text"].apply(clean_text)
df_tip["clean_text"] = df_tip["text"].apply(clean_text)

## 7. User Table — Elite Flag & Friends Count

In [18]:
df_user["is_elite"] = df_user["elite"].apply(lambda x: len(x) > 0 if isinstance(x, list) else False)
df_user["friends_count"] = df_user["friends"].apply(lambda x: len(x) if isinstance(x, list) else 0)

## 8. Checkin — Convert Date String to Count

In [19]:
df_checkin["checkin_count"] = df_checkin["date"].apply(lambda x: len(x.split(",")) if isinstance(x, str) else 0)

## 9. Save Processed Data

In [20]:
# ID-Based Sampling (relationship-driven, not independent)
RANDOM_STATE = 42

N_BUSINESS = 3000   # controls the size of everything downstream

# Step 1: Sample businesses
df_business_new = (
    df_business_flat
    .sample(n=N_BUSINESS, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

sampled_business_ids = set(df_business_new["business_id"])

# Step 2: Keep ALL related reviews, tips, and checkins
df_review_new = (
    df_review[df_review["business_id"].isin(sampled_business_ids)]
    .reset_index(drop=True)
)

df_tip_new = (
    df_tip[df_tip["business_id"].isin(sampled_business_ids)]
    .reset_index(drop=True)
)

df_checkin_new = (
    df_checkin[df_checkin["business_id"].isin(sampled_business_ids)]
    .reset_index(drop=True)
)

# Step 3: Get all users who wrote those reviews or tips
sampled_user_ids = (
    set(df_review_new["user_id"])
    .union(set(df_tip_new["user_id"]))
)

df_user_new = (
    df_user[df_user["user_id"].isin(sampled_user_ids)]
    .reset_index(drop=True)
)

# Print shapes
print("Post-sampling shapes (relationship-driven, joinable):")
for name, df in [
    ("business", df_business_new),
    ("checkin", df_checkin_new),
    ("tip", df_tip_new),
    ("review", df_review_new),
    ("user", df_user_new),
]:
    print(f"{name:10s}: {df.shape}")

Post-sampling shapes (relationship-driven, joinable):
business  : (3000, 59)
checkin   : (2645, 3)
tip       : (18528, 6)
review    : (138747, 10)
user      : (116220, 24)


In [21]:
df_business = df_business_new
df_checkin  = df_checkin_new
df_tip      = df_tip_new
df_review   = df_review_new
df_user     = df_user_new

In [32]:
df_business_flat.to_csv(f"{PROCESSED_DIR}/business_clean.csv", index=False)
df_checkin.to_csv(f"{PROCESSED_DIR}/checkin_clean.csv", index=False)
df_tip.to_csv(f"{PROCESSED_DIR}/tip_clean.csv", index=False)
df_review.to_csv(f"{PROCESSED_DIR}/review_clean.csv", index=False)
df_user.to_csv(f"{PROCESSED_DIR}/user_clean.csv", index=False)

print("All processed files saved as CSV to:", PROCESSED_DIR)

All processed files saved as CSV to: /Users/manjushwarkhairkar/GitHub/FoodRec/notebooks/data/processed


## Optional

In [35]:
import pandas as pd

df = pd.read_csv("/Users/manjushwarkhairkar/github/FoodRec/data/processed/user_clean.csv")
df.shape

# Sample 500 random rows (adjust n as needed)
df_sample = df.sample(n=500, random_state=42)

df_sample.to_csv("/Users/manjushwarkhairkar/github/FoodRec/data/processed/user_clean_sample.csv", index=False)

print(f"Original: {len(df)} rows")
print(f"Sampled: {len(df_sample)} rows")

Original: 116220 rows
Sampled: 500 rows
